In [8]:
import pandas as pd
INPUT = "HIGGS.csv"
OUTPUT = "HIGGS_cleaned.csv"
# Definiamo i nomi: la prima è la Label, le altre 28 sono Feature
column_names = ['Label'] + [f'feat_{i}' for i in range(1, 29)]

# Caricamento (se il file è senza header)
df = pd.read_csv(INPUT, header=None, names=column_names)
df.head()


,Label,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,feat_8,feat_9,...,feat_19,feat_20,feat_21,feat_22,feat_23,feat_24,feat_25,feat_26,feat_27,feat_28
0,1.0,0.869293,-0.635082,0.225690,0.327470,-0.689993,0.754202,-0.248573,-1.092064,0.000000,...,-0.010455,-0.045767,3.101961,1.353760,0.979563,0.978076,0.920005,0.721657,0.988751,0.876678
1,1.0,0.907542,0.329147,0.359412,1.497970,-0.313010,1.095531,-0.557525,-1.588230,2.173076,...,-1.138930,-0.000819,0.000000,0.302220,0.833048,0.985700,0.978098,0.779732,0.992356,0.798343
2,1.0,0.798835,1.470639,-1.635975,0.453773,0.425629,1.104875,1.282322,1.381664,0.000000,...,1.128848,0.900461,0.000000,0.909753,1.108330,0.985692,0.951331,0.803252,0.865924,0.780118
3,0.0,1.344385,-0.876626,0.935913,1.992050,0.882454,1.786066,-1.646778,-0.942383,0.000000,...,-0.678379,-1.360356,0.000000,0.946652,1.028704,0.998656,0.728281,0.869200,1.026736,0.957904
4,1.0,1.105009,0.321356,1.522401,0.882808,-1.205349,0.681466,-1.070464,-0.921871,0.000000,...,-0.373566,0.113041,0.000000,0.755856,1.361057,0.986610,0.838085,1.133295,0.872245,0.808487


In [10]:
import numpy as np
# --- 2. PULIZIA (Sanitizzazione) ---
print("🧹 Inizio pulizia valori...")

# Clip preventivo: costringiamo tutto nel range del float32 sicuro
# Questo impedisce l'errore "Input contains infinity" nei Worker
f32_min = np.finfo(np.float32).min
f32_max = np.finfo(np.float32).max

# Selezioniamo solo le feature (escludendo la label che è 0 o 1)
features = [c for c in df.columns if c != 'label']
df[features] = df[features].clip(lower=f32_min, upper=f32_max)

# Rimuoviamo eventuali NaN o Inf residui
df.replace([np.inf, -np.inf], np.nan, inplace=True)
original_len = len(df)
df.dropna(inplace=True)
print(f"   Righe rimosse (sporche): {original_len - len(df)}")

# --- 3. SHUFFLE E SPLIT ---
print("🔀 Mescolamento dati (Shuffle)...")
# Mescoliamo tutto il dataset. random_state=42 per riproducibilità
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df.to_csv(OUTPUT, index=False)

🧹 Inizio pulizia valori...
   Righe rimosse (sporche): 0
🔀 Mescolamento dati (Shuffle)...


In [20]:
import pandas as pd
import numpy as np
import time
import psutil
import os
import gc
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

df = pd.read_csv("HIGGS_cleaned.csv", dtype=np.float32)


# 2. SPLIT 80/20
print("✂️ Eseguendo split 80/20...")

# Separiamo Features (X) e Target (y)
X = df.drop(columns=['Label'])
y = df['Label']

# IMPORTANTE: Convertiamo y in int (0/1) se è float, per sicurezza
y = y.astype(int)

# Split casuale 80% Train, 20% Test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y  # Mantiene la proporzione di classi bilanciata
)

print(f"   -> Training Set: {X_train.shape[0]} righe")
print(f"   -> Test Set:     {X_test.shape[0]} righe")

# Liberiamo RAM eliminando il df originale
del df
gc.collect()

# 3. TRAINING (Random Forest)
print("\n🌲 Avvio Training (500 alberi, depth=20)...")
print("⚡️ n_jobs=-1 attivo: Le ventole del Mac partiranno al massimo!")

rf = RandomForestClassifier(
    n_estimators=50,     # Stesso setup del distribuito
    max_depth=20,         # Stesso setup del distribuito
    n_jobs=-1,            # Usa tutti i core
    verbose=1,            # Mostra i log
    random_state=193
)

start_train = time.time()
rf.fit(X_train, y_train)
end_train = time.time()

training_time = end_train - start_train
print(f"🏁 Training completato in: {training_time:.2f} secondi")

# 4. VALUTAZIONE
print("🔮 Inizio Inferenza su 2.2M righe...")
start_pred = time.time()
y_pred = rf.predict(X_test)
end_pred = time.time()

acc = accuracy_score(y_test, y_pred)

print("\n" + "="*40)
print(f"📊 RISULTATI LOCALE (HIGGS 80/20)")
print("="*40)
print(f"✅ Accuracy:        {acc*100:.4f}%")
print(f"⏱️ Tempo Training:  {training_time:.2f} s")
print(f"⏱️ Tempo Inferenza: {end_pred - start_pred:.2f} s")
print("="*40)

✂️ Eseguendo split 80/20...
   -> Training Set: 8800000 righe
   -> Test Set:     2200000 righe

🌲 Avvio Training (500 alberi, depth=20)...
⚡️ n_jobs=-1 attivo: Le ventole del Mac partiranno al massimo!


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=-1)]: Done  30 tasks      | elapsed:  3.8min
[Parallel(n_jobs=-1)]: Done  50 out of  50 | elapsed:  6.4min finished
[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.


🏁 Training completato in: 384.88 secondi
🔮 Inizio Inferenza su 2.2M righe...


[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    3.5s



📊 RISULTATI LOCALE (HIGGS 80/20)
✅ Accuracy:        74.2059%
⏱️ Tempo Training:  384.88 s
⏱️ Tempo Inferenza: 6.48 s


[Parallel(n_jobs=10)]: Done  50 out of  50 | elapsed:    6.4s finished
